In [5]:
import pandas as pd
import duckdb
import dlt
import numpy as np
import os

print("Libraries imported successfully")

Libraries imported successfully


In [10]:
# Read CSV 
path = "../data/sample.csv"
def read_data(path):
    df = pd.read_csv(path)
    return df

df = read_data(path) 
df.head()

,order_id,product,category,region,quantity,price,revenue
0,1.0,Laptop,Electronics,North,2,50000,100000
1,2.0,Phone,Electronics,South,3,20000,60000
2,3.0,Desk,Furniture,West,1,10000,10000
3,4.0,Chair,Furniture,East,4,3000,11000
4,5.0,Monitor,Electronics,North,2,15000,30000


In [19]:
def transform_data(df):
    #to remove entries with null values 
    df["remarks"]=np.where(df["order_id"].isna(),"invalid","valid")

    removed = df[df["remarks"]=="invalid"]
    df = df[df["remarks"]=="valid"]
    
    df["order_id"]=df["order_id"].astype("int")
    df["order_id"]=df["order_id"].astype("string")

    columns = ["quantity","price","revenue"]

    for col in columns :
        df[col]=df[col].astype("float")
        df[col]=df[col].abs()

    columns = ["product","category","region"]

    for col in columns :
        df[col]=df[col].fillna("Unknown")
    
    #print(df)

    df["remarks"]=np.where(abs((df["quantity"]*df["price"])-df["revenue"])< 1 , "valid", "invalid revenue")

    invalid_revenue = df[df["remarks"]=="invalid revenue"]   

    #print(df)

    df = df[df["remarks"]=="valid"]

    print("Data transformations completed successfully!")

    removed= pd.concat([removed,invalid_revenue],ignore_index=True)
    print("\n Removed Rows")
    print(removed)
    print("\n Cleaned Data")
    print(df)

    removed.to_csv("../outputs/removed_entries.csv")
    return df
    
cleaned_data=transform_data(df)
cleaned_data = cleaned_data.to_dict(orient="records")

Data transformations completed successfully!

 Removed Rows
  order_id  product     category region  quantity   price  revenue  \
0      NaN  Printer  Electronics  South       2.0  8000.0  16000.0   
1        4    Chair    Furniture   East       4.0  3000.0  11000.0   

           remarks  
0          invalid  
1  invalid revenue  

 Cleaned Data
  order_id   product     category   region  quantity    price   revenue  \
0        1    Laptop  Electronics    North       2.0  50000.0  100000.0   
1        2     Phone  Electronics    South       3.0  20000.0   60000.0   
2        3      Desk    Furniture     West       1.0  10000.0   10000.0   
4        5   Monitor  Electronics    North       2.0  15000.0   30000.0   
5        6    Tablet  Electronics    South       1.0  25000.0   25000.0   
6        7      Sofa    Furniture     West       1.0  40000.0   40000.0   
7        8  Keyboard  Electronics  Unknown       5.0   1000.0    5000.0   
8        9     Mouse  Electronics    North      10.

In [20]:
#isse humari ek data pipeline ban jaegi --> 
def load_data_db(cleaned_data):
    data_load_pipeline = dlt.pipeline(
    pipeline_name = "data_load_pipeline",
    destination = "duckdb", #ye aapka db type hoga :-> postgres/duckdb/snowflake/bigquery
    dataset_name = "sales_data"
    )

    print("Pipeline Created")

    info=data_load_pipeline.run(
    cleaned_data, 
    table_name = "sales",
    write_disposition = "replace"
    )

    print("Data Pushed")

    #lets check if data is pushed

    db_path = data_load_pipeline.sql_client().credentials.database
    con = duckdb.connect(db_path)
    
    loaded_table=con.execute("Select * from sales_data.sales;").fetchdf()

    print("Data Loaded successfully")
    print(loaded_table.head())
    
    return db_path


In [21]:
db_path = load_data_db(cleaned_data)

Pipeline Created
Data Pushed
Data Loaded successfully
  order_id  product     category region  quantity    price   revenue remarks  \
0        1   Laptop  Electronics  North       2.0  50000.0  100000.0   valid   
1        2    Phone  Electronics  South       3.0  20000.0   60000.0   valid   
2        3     Desk    Furniture   West       1.0  10000.0   10000.0   valid   
3        5  Monitor  Electronics  North       2.0  15000.0   30000.0   valid   
4        6   Tablet  Electronics  South       1.0  25000.0   25000.0   valid   

        _dlt_load_id         _dlt_id  
0  1781239423.732468  gbf2keF6KWqycA  
1  1781239423.732468  oGPknZP1aAqNNA  
2  1781239423.732468  kmSveXp/2wFz6A  
3  1781239423.732468  MDArF+pdflQdBg  
4  1781239423.732468  S0WwZRFRLqPi8g  


In [22]:
def analyse_data(db_path):
    con = duckdb.connect(db_path)
    print(db_path)

    print("\nSummary Statistics:")
    summary_table= con.execute(
    """select count(*) as total_orders,
    sum(quantity) as total_quantity_sold,
    sum(revenue) as total_revenue, 
    round(avg(price),2) as avg_ticket
    from sales_data.sales; """).fetchdf()

    summary_table.to_csv("../outputs/summary_statistics.csv",index=False)

    print('\n',summary_table)

    print("\nCategory Wise Statistics:")
    category_data= con.execute(
    """select category,count(*) as total_orders,
    sum(quantity) as total_quantity_sold,
    sum(revenue) as total_revenue, 
    round(avg(price),2) as avg_ticket
    from sales_data.sales
    group by 1
    ; """).fetchdf()

    print('\n',category_data)

    category_data.to_csv("../outputs/sales_by_category.csv",index=False)

    print("\nRegion Wise Statistics:")
    region_data= con.execute(
    """select region,count(*) as total_orders,
    sum(quantity) as total_quantity_sold,
    sum(revenue) as total_revenue, 
    round(avg(price),2) as avg_ticket
    from sales_data.sales
    group by 1
    ; """).fetchdf()

    print('\n',region_data)

    region_data.to_csv("../outputs/sales_by_region.csv",index=False)

    outputs=["../outputs/sales_by_region.csv","../outputs/sales_by_category.csv","../outputs/summary_statistics.csv"]

    for path in outputs:
        if os.path.exists(path):
            print(f"File downloaded : {path}")
    
    return

analyse_data(db_path)

/workspaces/Local-Data-Engineering-Environment-with-dlt-DuckDB-Jupyter/notebooks/data_load_pipeline.duckdb

Summary Statistics:

    total_orders  total_quantity_sold  total_revenue  avg_ticket
0             9                 27.0       291000.0    18833.33

Category Wise Statistics:

       category  total_orders  total_quantity_sold  total_revenue  avg_ticket
0  Electronics             7                 25.0       241000.0    17071.43
1    Furniture             2                  2.0        50000.0    25000.00

Region Wise Statistics:

     region  total_orders  total_quantity_sold  total_revenue  avg_ticket
0    North             3                 14.0       135000.0    21833.33
1    South             3                  6.0       101000.0    17666.67
2     West             2                  2.0        50000.0    25000.00
3  Unknown             1                  5.0         5000.0     1000.00
File downloaded : ../outputs/sales_by_region.csv
File downloaded : ../outputs/sales_by_cat